In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
data = spark.read.csv(
    "E:/Cluster Computing/Dataset/ca3_task1_student_data.csv",
    header = True,
    inferSchema = True
)

In [4]:
data.show()

+---------+----------+----------+-------------+------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|
+---------+----------+----------+-------------+------+
|      101|       2.0|        70|            1|  Pass|
|      102|       3.5|        85|            3|  Pass|
|      103|       1.0|        60|            0|  Fail|
|      104|       5.0|        95|            5|  Pass|
|      105|       4.0|        90|            4|  Pass|
|      106|       2.5|        75|            2|  Fail|
+---------+----------+----------+-------------+------+



In [5]:
data.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- StudyHours: double (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- PracticeTests: integer (nullable = true)
 |-- Result: string (nullable = true)



In [6]:
from pyspark.ml.feature import StringIndexer

In [7]:
indexer = StringIndexer(
    inputCol = "Result",
    outputCol = "ResultIndexed"
)
data1 = indexer.fit(data)
data = data1.transform(data)

In [8]:
data.select("Result", "ResultIndexed").show()

+------+-------------+
|Result|ResultIndexed|
+------+-------------+
|  Pass|          0.0|
|  Pass|          0.0|
|  Fail|          1.0|
|  Pass|          0.0|
|  Pass|          0.0|
|  Fail|          1.0|
+------+-------------+



In [9]:
data.show()

+---------+----------+----------+-------------+------+-------------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|ResultIndexed|
+---------+----------+----------+-------------+------+-------------+
|      101|       2.0|        70|            1|  Pass|          0.0|
|      102|       3.5|        85|            3|  Pass|          0.0|
|      103|       1.0|        60|            0|  Fail|          1.0|
|      104|       5.0|        95|            5|  Pass|          0.0|
|      105|       4.0|        90|            4|  Pass|          0.0|
|      106|       2.5|        75|            2|  Fail|          1.0|
+---------+----------+----------+-------------+------+-------------+



In [10]:
# Independent Columns - StudyHours, Attendance, PracticeTests
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols = ["StudyHours", "Attendance", "PracticeTests"],
    outputCol = "features"
)
data = assembler.transform(data)

In [11]:
data.show()

+---------+----------+----------+-------------+------+-------------+--------------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|ResultIndexed|      features|
+---------+----------+----------+-------------+------+-------------+--------------+
|      101|       2.0|        70|            1|  Pass|          0.0|[2.0,70.0,1.0]|
|      102|       3.5|        85|            3|  Pass|          0.0|[3.5,85.0,3.0]|
|      103|       1.0|        60|            0|  Fail|          1.0|[1.0,60.0,0.0]|
|      104|       5.0|        95|            5|  Pass|          0.0|[5.0,95.0,5.0]|
|      105|       4.0|        90|            4|  Pass|          0.0|[4.0,90.0,4.0]|
|      106|       2.5|        75|            2|  Fail|          1.0|[2.5,75.0,2.0]|
+---------+----------+----------+-------------+------+-------------+--------------+



In [12]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler(
    inputCol = "features",
    outputCol = "scaledFeatures"
)
scaler_model = scaler.fit(data)
data = scaler_model.transform(data)

In [13]:
data.show()

+---------+----------+----------+-------------+------+-------------+--------------+--------------------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|ResultIndexed|      features|      scaledFeatures|
+---------+----------+----------+-------------+------+-------------+--------------+--------------------+
|      101|       2.0|        70|            1|  Pass|          0.0|[2.0,70.0,1.0]|[1.38013111868470...|
|      102|       3.5|        85|            3|  Pass|          0.0|[3.5,85.0,3.0]|[2.41522945769823...|
|      103|       1.0|        60|            0|  Fail|          1.0|[1.0,60.0,0.0]|[0.69006555934235...|
|      104|       5.0|        95|            5|  Pass|          0.0|[5.0,95.0,5.0]|[3.45032779671177...|
|      105|       4.0|        90|            4|  Pass|          0.0|[4.0,90.0,4.0]|[2.76026223736941...|
|      106|       2.5|        75|            2|  Fail|          1.0|[2.5,75.0,2.0]|[1.72516389835588...|
+---------+----------+----------+-------------+------+-

In [14]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

In [15]:
data.count()

6

In [16]:
train_data.count()

5

In [17]:
test_data.count()

1

In [18]:
from pyspark.ml.classification import DecisionTreeClassifier

In [19]:
dt = DecisionTreeClassifier(
    featuresCol = "features", # Independent Columns
    labelCol = "ResultIndexed" # Dependent Columns
)

In [20]:
model = dt.fit(train_data)

In [21]:
predictions = model.transform(test_data)

In [22]:
predictions.show()

+---------+----------+----------+-------------+------+-------------+--------------+--------------------+-------------+-----------+----------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|ResultIndexed|      features|      scaledFeatures|rawPrediction|probability|prediction|
+---------+----------+----------+-------------+------+-------------+--------------+--------------------+-------------+-----------+----------+
|      103|       1.0|        60|            0|  Fail|          1.0|[1.0,60.0,0.0]|[0.69006555934235...|    [1.0,0.0]|  [1.0,0.0]|       0.0|
+---------+----------+----------+-------------+------+-------------+--------------+--------------------+-------------+-----------+----------+



In [23]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol = "ResultIndexed",
    predictionCol = "prediction",
    metricName = "accuracy"
)
accuracy = evaluator.evaluate(predictions)
accuracy

0.0

In [24]:
# Number of correct and incorrect predictions:
from pyspark.sql.functions import col

predictions.select(
    (col("ResultIndexed") == col("prediction")).cast("int").alias("Correct")
).groupBy("Correct").count().show()

+-------+-----+
|Correct|count|
+-------+-----+
|      0|    1|
+-------+-----+



In [25]:
predictions.groupBy("ResultIndexed", "prediction") \
    .count() \
    .orderBy("ResultIndexed", "prediction") \
    .show()

+-------------+----------+-----+
|ResultIndexed|prediction|count|
+-------------+----------+-----+
|          1.0|       0.0|    1|
+-------------+----------+-----+



In [26]:
for metric in ["f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="ResultIndexed",
        predictionCol="prediction",
        metricName=metric
    )
    print(metric, ":", evaluator.evaluate(predictions))

f1 : 0.0
weightedPrecision : 0.0
weightedRecall : 0.0
